# Load `COLLABORATED_WITH` Edges

The first **derived** relationship in the graph. Until now every edge was loaded from a source file; this one is *computed from the graph itself* via a single Cypher query.

Two actors are "collaborators" if they appeared together in at least one movie (within our top-10-billing scope from FEATURES_ACTOR). The whole computation is server-side — no Python compute, no edge readers in `etl.py`.

See [`docs/07_collaborated_with.md`](../docs/07_collaborated_with.md) for the design decisions, the MERGE gotcha, and the pedagogical walkthrough.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from graph import run_query, run_write_query, close_driver

## Optional Surgical Teardown

Unlike the optional teardowns in 04/05 (which wipe everything), this one is **label-scoped** — it deletes only `:COLLABORATED_WITH` edges. No nodes are touched, and no other relationship types are affected. The rest of the graph stays intact, so 04/05/06 do **not** need to re-run after teardown.

The `MERGE` + `SET` pattern in the next cell already makes the load idempotent, so teardown is only needed on first run or to refresh after a `:FEATURES_ACTOR` change.

In [2]:
# Uncomment to wipe existing COLLABORATED_WITH edges before re-running.
# Safe — label-scoped, only touches this edge type.
# run_write_query("MATCH ()-[r:COLLABORATED_WITH]->() DELETE r")

## Compute and Load `COLLABORATED_WITH`

One query, four moving parts:

1. **MATCH** the two-hop pattern `(actor)←(movie)→(actor)` — every pair of actors that share a movie.
2. **WHERE** the canonical-ordering filter `a1.name < a2.name` — guarantees one row per pair (decision 1: store single canonical direction).
3. **WITH ... count(m)** aggregates per pair — counts how many distinct shared movies (decision 2: store `movie_count`).
4. **MERGE + SET** writes the edge with the count. `MERGE` matches on labels + direction only; `SET` writes the property separately — see [`docs/07_collaborated_with.md`](../docs/07_collaborated_with.md) for why inlining the property in MERGE is a bug.

In [3]:
compute_query = """
MATCH (a1:Actor)<-[:FEATURES_ACTOR]-(m:Movie)-[:FEATURES_ACTOR]->(a2:Actor)
WHERE a1.name < a2.name
WITH a1, a2, count(m) AS movie_count
MERGE (a1)-[r:COLLABORATED_WITH]->(a2)
SET r.movie_count = movie_count
"""

run_write_query(compute_query)
print("COLLABORATED_WITH edges computed.")

COLLABORATED_WITH edges computed.


## Verification

### Edge count

In [4]:
result = run_query("MATCH ()-[r:COLLABORATED_WITH]->() RETURN count(r) AS total")
print(f"Total COLLABORATED_WITH edges: {result[0]['total']:,}")

Total COLLABORATED_WITH edges: 70,139


### Summary — edge counts by relationship type

After this notebook runs cleanly, the summary should report **7 relationship types** (the 6 from 04/05/06 + `COLLABORATED_WITH`).

In [5]:
summary = run_query("""
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, count(r) AS count
ORDER BY count DESC
""")
for row in summary:
    print(f"{row['relationshipType']:<25} {row['count']:>10,}")

RATED                        100,000
COLLABORATED_WITH             70,139
FEATURES_ACTOR                16,243
IN_GENRE                       2,893
DIRECTED_BY                    1,790
HAS_OCCUPATION                   943
HAS_GENDER                       943


## Spot-Checks

### Spot-check 1 — Top 10 collaborator pairs by `movie_count`

Sanity check: the top results should be plausible. Expect repeat-pairing actors from the ml-100k era — Hanks/Ryan-types, De Niro/Pesci-types, Scorsese regulars. If the top result is two unknown actors with `movie_count = 1`, something is wrong.

We use undirected match `-[r]-` because the stored direction (alphabetical canonical) is semantically meaningless. The `WHERE a1.name < a2.name` filter dedups — without it, each pair would appear twice (once per binding orientation).

In [6]:
top_pairs = run_query("""
MATCH (a1:Actor)-[r:COLLABORATED_WITH]-(a2:Actor)
WHERE a1.name < a2.name
RETURN a1.name AS actor1, a2.name AS actor2, r.movie_count AS shared_movies
ORDER BY shared_movies DESC LIMIT 10
""")
for row in top_pairs:
    print(f"{row['actor1']:<25} {row['actor2']:<25} {row['shared_movies']:>3}")

James Doohan              William Shatner             7
George Takei              William Shatner             6
DeForest Kelley           George Takei                6
DeForest Kelley           Walter Koenig               6
George Takei              Nichelle Nichols            6
George Takei              James Doohan                6
DeForest Kelley           William Shatner             6
DeForest Kelley           James Doohan                6
DeForest Kelley           Nichelle Nichols            6
George Takei              Walter Koenig               6


### Spot-check 2 — Tom Hanks's top collaborators

A recognizable actor with substantial ml-100k coverage (Big, Forrest Gump, Apollo 13, Toy Story, Sleepless in Seattle, Philadelphia, A League of Their Own, You've Got Mail). His top collaborators should include co-stars from those films.

If the result is empty, Tom Hanks isn't an `:Actor` node in our graph — pick another well-known actor and re-run.

In [7]:
hanks_collabs = run_query("""
MATCH (target:Actor {name: "Tom Hanks"})-[r:COLLABORATED_WITH]-(other:Actor)
RETURN other.name AS collaborator, r.movie_count AS shared_movies
ORDER BY shared_movies DESC LIMIT 10
""")
if hanks_collabs:
    for row in hanks_collabs:
        print(f"{row['collaborator']:<30} {row['shared_movies']:>3}")
else:
    print("Tom Hanks not found in graph.")
    print("Run: MATCH (a:Actor) RETURN a.name LIMIT 20 — pick another actor and retry.")

Gary Sinise                      2
Obba Babatundé                   2
Xander Berkeley                  1
Antonio Banderas                 1
Barbara Garrick                  1
Bill Paxton                      1
Brett Cullen                     1
Bill Pullman                     1
Charles Napier                   1
Annie Potts                      1


### Spot-check 3 — Triangle test (3 mutual collaborators)

**This is the pedagogically strongest query in the notebook.** Three actors who have *all* worked with each other — A↔B, B↔C, and A↔C.

In SQL this would be three self-joins on a movie_cast table with a `WHERE a.id < b.id AND b.id < c.id` trim — verbose, slow, and hard to read. In Cypher it's one pattern: traverse the triangle. The `<` chain trims to unique triples (avoids 6 permutations per real triangle).

This is the canonical "easy in graph, brutal in SQL" example.

In [8]:
triangles = run_query("""
MATCH (a:Actor)-[:COLLABORATED_WITH]-(b:Actor)-[:COLLABORATED_WITH]-(c:Actor)-[:COLLABORATED_WITH]-(a:Actor)
WHERE a.name < b.name AND b.name < c.name
RETURN a.name AS actor_a, b.name AS actor_b, c.name AS actor_c LIMIT 10
""")
for row in triangles:
    print(f"{row['actor_a']:<22} {row['actor_b']:<22} {row['actor_c']:<22}")

Aidan Quinn            Al Pacino              Alec Baldwin          
Al Pacino              Alan Arkin             Alec Baldwin          
Alec Guinness          Aleksandr Bureyev      Aleksandr Pyatkov     
Adam Sandler           Alexis Arquette        Allen Covert          
Alan Cox               Alan Rickman           Alun Armstrong        
Alan Alda              Allison Janney         Amo Gulinello         
Aidan Quinn            Alice Drummond         Amy Sedaris           
Aleksandr Kaydanovskiy Alisa Freyndlikh       Anatoliy Solonitsyn   
Aisling O'Sullivan     Alan Boyle             Andrew Fullerton      
Alan Arkin             Ally Walker            Andy Garcia           


## Idempotency Note

Same `MERGE` guarantee as 04, 05, and 06: re-running the compute cell produces identical edges. The `SET r.movie_count = movie_count` clause also refreshes the count on each run — so if a future `:FEATURES_ACTOR` change adds a movie between two existing collaborators, the count updates correctly on re-run (no stale data).

In [9]:
close_driver()